In [3]:
from litellm import completion
from typing import List
import json
import litellm
from tqdm import tqdm
# litellm.set_verbose=True

In [ ]:
response = completion(
    # model="ollama/llama3.2",
    model="ollama/adrienbrault/nous-hermes2pro-llama3-8b:q8_0",
    messages=[{ "content": f"You are a name entity recognition system you will extract entities related to medicine, specially for diabetes information that I will give you in the following prompt"
               f"and you will extract all meaningful entities from the given discharge summary"
               f"You will return only JSON outputs","role": "user"}],
    # json=True
)
print(response)


In [11]:
def format_entities(ent_list:List[str]) -> str:
    return "\n\n".join([e for e in ent_list])

In [12]:
import json

with open('diabetes_data_preprocessed.json') as f:
    chunks = json.load(f)
print(chunks[0])

{'context': 'Patient ID: 10000980 Patient Race: BLACK/AFRICAN AMERICAN  Admission Type: OBSERVATION ADMIT  Diagnosis: type 2 diabetes mellitus with diabetic chronic kidney disease (ICD Code: E1122)  Medication: Insulin  Discharge Summary: name N/A unit no N/A admission date N/A discharge date N/A date of birth N/A sex f service medicine allergies no known allergies adverse drug reactions attending N/A chief complaint shortness of breath major surgical or invasive procedure none history of present illness patient is a N/A with history of coronary artery disease cb ischemic mr N/A des to lcx N/A tte N/A with mild regional lv systolic dysfunction heart failure with preserved ejection fraction lvef 50 N/A peripheral vascular disease chronic kidney disease stage iv prior unprovoked dvt cb severe ugib while on ac htn dyslipidemia and t2dm who presents with several days of shortness of breath patients says that she first noticed rather acute onset dyspnea starting N/A when trying to walk up t

In [15]:
system_message = """
Extract all meaningful entities from the given discharge summary, focusing on the following topics:
- Patient Demographics and Identifiers
- Primary and Secondary Diagnoses
- Medications and Treatments (both at admission and discharge)
- Discharge Instructions
- Additional Notes (including tests, mental status, and other relevant observations)

Return all extracted entities as a JSON array. Each object in the array should represent a meaningful entity and contain two fields:
1. "entity": The extracted entity (e.g., names, conditions, medications, dosages, instructions, dates, etc.)
2. "type": The type of the entity (e.g., "name", "diagnosis", "medication", "dosage", "instruction", "test", "mental status", etc.)

Ensure each entity is unique, meaningful, and relevant to the discharge summary.

Example output:
[
  {"entity": "10000980", "type": "Patient_ID"},
  {"entity": "BLACK/AFRICAN AMERICAN", "type": "Patient_race"},
  {"entity": "Acute on Chronic Diastolic Congestive Heart Failure", "type": "diagnosis"},
  {"entity": "Hypertension", "type": "diagnosis"},
  {"entity": "Allopurinol", "type": "medication"},
  {"entity": "80 mg", "type": "dosage"},
  {"entity": "qpm", "type": "dosage frequency"},
  {"entity": "Monitor and limit salt intake", "type": "discharge_instruction"},
  {"entity": "clear and coherent", "type": "mental_status"},
  {"entity": "chest X-ray", "type": "test"}
]

Return **strictly JSON format** for all extracted entities.

 """

text = chunks[11]['context']

user_message = "Context: {text}\n\nTriples:"
response = completion(
  # api_key=OPENAI_API_KEY,
  model="ollama/adrienbrault/nous-hermes2pro-llama3-8b:q8_0",
  # model="ollama/llama3.2:latest",
  # model="gpt-3.5-turbo",
  messages=[{"content": system_message,"role": "system"}, {"content": user_message.format(text=text),"role": "user"}],
  max_tokens=7000,
  temperature=0.3,
  # format = "json",
  api_base="http://localhost:11434"
)
print(response.choices[0].message.content)
# triples = json.loads(response.choices[0].message.content)
# triples

18:27:18 - LiteLLM:WARNING: utils.py:299 - `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.




Request to litellm:
litellm.completion(model='ollama/adrienbrault/nous-hermes2pro-llama3-8b:q8_0', messages=[{'content': '\nExtract all meaningful entities from the given discharge summary, focusing on the following topics:\n- Patient Demographics and Identifiers\n- Primary and Secondary Diagnoses\n- Medications and Treatments (both at admission and discharge)\n- Discharge Instructions\n- Additional Notes (including tests, mental status, and other relevant observations)\n\nReturn all extracted entities as a JSON array. Each object in the array should represent a meaningful entity and contain two fields:\n1. "entity": The extracted entity (e.g., names, conditions, medications, dosages, instructions, dates, etc.)\n2. "type": The type of the entity (e.g., "name", "diagnosis", "medication", "dosage", "instruction", "test", "mental status", etc.)\n\nEnsure each entity is unique, meaningful, and relevant to the discharge summary.\n\nExample output:\n[\n  {"entity": "10000980", "type": "Pat

In [17]:
system_message = """
Extract all the relationships between the following entities ONLY based on the given context.
Break the Discharge Summary: into smaller pieces of triplets and remove any '\' if any triplet contains it.
Return a list of JSON objects. For example:

[{"subject": "Patient ID: 10000980",
  "relationship": "is",
  "object": "Race: BLACK/AFRICAN AMERICAN"},
  {"subject": "Patient ID: 10000980",
  "relationship": "has",
  "object": "Admission Type: OBSERVATION ADMIT"},
  {"subject": "Patient ID: 10000980",
  "relationship": "has",
  "object": "Diagnosis: type 2 diabetes mellitus with diabetic chronic kidney disease (ICD Code: E1122)"},
  {"subject": "Patient ID: 10000980",
  "relationship": "receives",
  "object": "Medication: Insulin"},
  {"subject": "Patient ID: 10000980",
  "relationship": "has", "object":
  "Chief Complaint: shortness of breath"},
  {"subject": "Patient ID: 10000980",
  "relationship": "presents with",
  "object": "several days of shortness of breath"},
  {"subject": "Patient ID: 10000980",
  "relationship": "has",
  "object": "Weight gain over the past week (7lbs)"},
  {"subject": "Patient ID: 10000980",
  "relationship": "is taking",
  "object": "Torsemide 40mg qd"},
  {"subject": "Patient ID: 10000980",
  "relationship": "has no known",
  "object": "Allergies"},]


- ONLY return triples and nothing else. None of 'subject', 'relationship' and 'object' can be empty.

 """

text = chunks[0]['context']

user_message = "Context: {text}\n\nTriples:"
response = completion(
  # api_key=OPENAI_API_KEY,
  model="ollama/adrienbrault/nous-hermes2pro-llama3-8b:q8_0",
  # model="ollama/llama3.2:latest",
  # model="gpt-3.5-turbo",
  messages=[{"content": system_message,"role": "system"}, {"content": user_message.format(text=text),"role": "user"}],
  max_tokens=7500,
  # format = "json",
  api_base="http://localhost:11434",
  temperature=0.3
)
# print(response.choices[0].message.content)
triples = json.loads(response.choices[0].message.content)
print(triples)
len(triples)

18:28:49 - LiteLLM:WARNING: utils.py:299 - `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.




Request to litellm:
litellm.completion(model='ollama/adrienbrault/nous-hermes2pro-llama3-8b:q8_0', messages=[{'content': '\nExtract all the relationships between the following entities ONLY based on the given context.\nBreak the Discharge Summary: into smaller pieces of triplets and remove any \'\' if any triplet contains it.\nReturn a list of JSON objects. For example:\n\n[{"subject": "Patient ID: 10000980",\n  "relationship": "is",\n  "object": "Race: BLACK/AFRICAN AMERICAN"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has",\n  "object": "Admission Type: OBSERVATION ADMIT"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has",\n  "object": "Diagnosis: type 2 diabetes mellitus with diabetic chronic kidney disease (ICD Code: E1122)"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "receives",\n  "object": "Medication: Insulin"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has", "object":\n  "Chief Complaint: shortness of breath"},

9

Returned custom cost for model=ollama/adrienbrault/nous-hermes2pro-llama3-8b:q8_0 - prompt_tokens_cost_usd_dollar: 0, completion_tokens_cost_usd_dollar: 0


In [ ]:
import time

errors = []
all_triples = []

# for i in tqdm(range(2)):
for i in tqdm(range(len(chunks))):
    try:
        text = chunks[i]['context']

        user_message = "Context: {text}\n\nTriples:"
        response = completion(
        # api_key=OPENAI_API_KEY,
        model="ollama/adrienbrault/nous-hermes2pro-llama3-8b:q8_0",
        # model="ollama/llama3.2:latest",
        # model="gpt-3.5-turbo",
        messages=[{"content": system_message,"role": "system"}, {"content": user_message.format(text=text),"role": "user"}],
        max_tokens=7500,
        # format = "json",
        api_base="http://localhost:11434",
        temperature=0.3
        )
        triples = json.loads(response.choices[0].message.content)
        all_triples.append(triples)
        time.sleep(3)
    except Exception as e:
        print(f"Error for chunk {i}, {e}")
        errors.append(response.choices[0].message.content)
        all_triples.append([])

  0%|                                                                                                                                                                                   | 0/525 [00:00<?, ?it/s]18:43:59 - LiteLLM:WARNING: utils.py:299 - `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.




Request to litellm:
litellm.completion(model='ollama/adrienbrault/nous-hermes2pro-llama3-8b:q8_0', messages=[{'content': '\nExtract all the relationships between the following entities ONLY based on the given context.\nBreak the Discharge Summary: into smaller pieces of triplets and remove any \'\' if any triplet contains it.\nReturn a list of JSON objects. For example:\n\n[{"subject": "Patient ID: 10000980",\n  "relationship": "is",\n  "object": "Race: BLACK/AFRICAN AMERICAN"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has",\n  "object": "Admission Type: OBSERVATION ADMIT"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has",\n  "object": "Diagnosis: type 2 diabetes mellitus with diabetic chronic kidney disease (ICD Code: E1122)"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "receives",\n  "object": "Medication: Insulin"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has", "object":\n  "Chief Complaint: shortness of breath"},

  0%|▎                                                                                                                                                                        | 1/525 [00:15<2:19:09, 15.94s/it]18:44:15 - LiteLLM:WARNING: utils.py:299 - `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.




Request to litellm:
litellm.completion(model='ollama/adrienbrault/nous-hermes2pro-llama3-8b:q8_0', messages=[{'content': '\nExtract all the relationships between the following entities ONLY based on the given context.\nBreak the Discharge Summary: into smaller pieces of triplets and remove any \'\' if any triplet contains it.\nReturn a list of JSON objects. For example:\n\n[{"subject": "Patient ID: 10000980",\n  "relationship": "is",\n  "object": "Race: BLACK/AFRICAN AMERICAN"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has",\n  "object": "Admission Type: OBSERVATION ADMIT"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has",\n  "object": "Diagnosis: type 2 diabetes mellitus with diabetic chronic kidney disease (ICD Code: E1122)"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "receives",\n  "object": "Medication: Insulin"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has", "object":\n  "Chief Complaint: shortness of breath"},

  0%|▋                                                                                                                                                                        | 2/525 [00:32<2:20:32, 16.12s/it]18:44:31 - LiteLLM:WARNING: utils.py:299 - `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.




Request to litellm:
litellm.completion(model='ollama/adrienbrault/nous-hermes2pro-llama3-8b:q8_0', messages=[{'content': '\nExtract all the relationships between the following entities ONLY based on the given context.\nBreak the Discharge Summary: into smaller pieces of triplets and remove any \'\' if any triplet contains it.\nReturn a list of JSON objects. For example:\n\n[{"subject": "Patient ID: 10000980",\n  "relationship": "is",\n  "object": "Race: BLACK/AFRICAN AMERICAN"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has",\n  "object": "Admission Type: OBSERVATION ADMIT"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has",\n  "object": "Diagnosis: type 2 diabetes mellitus with diabetic chronic kidney disease (ICD Code: E1122)"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "receives",\n  "object": "Medication: Insulin"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has", "object":\n  "Chief Complaint: shortness of breath"},

  1%|█▎                                                                                                                                                                       | 4/525 [01:07<2:31:11, 17.41s/it]18:45:07 - LiteLLM:WARNING: utils.py:299 - `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.




Request to litellm:
litellm.completion(model='ollama/adrienbrault/nous-hermes2pro-llama3-8b:q8_0', messages=[{'content': '\nExtract all the relationships between the following entities ONLY based on the given context.\nBreak the Discharge Summary: into smaller pieces of triplets and remove any \'\' if any triplet contains it.\nReturn a list of JSON objects. For example:\n\n[{"subject": "Patient ID: 10000980",\n  "relationship": "is",\n  "object": "Race: BLACK/AFRICAN AMERICAN"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has",\n  "object": "Admission Type: OBSERVATION ADMIT"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has",\n  "object": "Diagnosis: type 2 diabetes mellitus with diabetic chronic kidney disease (ICD Code: E1122)"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "receives",\n  "object": "Medication: Insulin"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has", "object":\n  "Chief Complaint: shortness of breath"},

  1%|█▌                                                                                                                                                                       | 5/525 [01:23<2:25:08, 16.75s/it]18:45:22 - LiteLLM:WARNING: utils.py:299 - `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.




Request to litellm:
litellm.completion(model='ollama/adrienbrault/nous-hermes2pro-llama3-8b:q8_0', messages=[{'content': '\nExtract all the relationships between the following entities ONLY based on the given context.\nBreak the Discharge Summary: into smaller pieces of triplets and remove any \'\' if any triplet contains it.\nReturn a list of JSON objects. For example:\n\n[{"subject": "Patient ID: 10000980",\n  "relationship": "is",\n  "object": "Race: BLACK/AFRICAN AMERICAN"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has",\n  "object": "Admission Type: OBSERVATION ADMIT"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has",\n  "object": "Diagnosis: type 2 diabetes mellitus with diabetic chronic kidney disease (ICD Code: E1122)"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "receives",\n  "object": "Medication: Insulin"},\n  {"subject": "Patient ID: 10000980",\n  "relationship": "has", "object":\n  "Chief Complaint: shortness of breath"},

In [ ]:
output_file = "triples.json"
json_data = json.dumps(all_triples, indent=4)
with open(output_file, "w") as file:
    file.write(json_data)

In [4]:
input_file = "triples.json"
with open(input_file, "r") as file:
    all_triples = json.load(file)

type(all_triples)
all_triples

[[{'subject': 'Patient ID: 10000980',
   'relationship': 'is',
   'object': 'Race: BLACK/AFRICAN AMERICAN'},
  {'subject': 'Patient ID: 10000980',
   'relationship': 'has',
   'object': 'Admission Type: OBSERVATION ADMIT'},
  {'subject': 'Patient ID: 10000980',
   'relationship': 'has',
   'object': 'Diagnosis: type 2 diabetes mellitus with diabetic chronic kidney disease (ICD Code: E1122)'},
  {'subject': 'Patient ID: 10000980',
   'relationship': 'receives',
   'object': 'Medication: Insulin'},
  {'subject': 'Patient ID: 10000980',
   'relationship': 'has',
   'object': 'Chief Complaint: shortness of breath'},
  {'subject': 'Patient ID: 10000980',
   'relationship': 'presents with',
   'object': 'several days of shortness of breath'},
  {'subject': 'Patient ID: 10000980',
   'relationship': 'has',
   'object': 'Weight gain over the past week (7lbs)'},
  {'subject': 'Patient ID: 10000980',
   'relationship': 'is taking',
   'object': 'Torsemide 40mg qd'},
  {'subject': 'Patient ID: 10

In [5]:
import csv



# Step 1: Extract unique nodes
nodes = set()
for triple_list in all_triples:
    for triplet in triple_list:
        # Add "subject" if it exists
        if "subject" in triplet:
            nodes.add((triplet["subject"], "Entity"))
        # Add "object" if it exists
        if "object" in triplet:
            nodes.add((triplet["object"], "Entity"))

# Step 2: Write nodes.csv
with open("nodes.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "label"])
    writer.writerows(nodes)

# Step 3: Extract relationships
edges = []
for triple_list in all_triples:
    for triplet in triple_list:
        if "subject" in triplet and "object" in triplet and "relationship" in triplet:
            edges.append((triplet["subject"], triplet["object"], triplet["relationship"]))

# Step 4: Write edges.csv
with open("edges.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["source", "target", "relationship"])
    writer.writerows(edges)

print("Nodes and edges files generated successfully.")


Nodes and edges files generated successfully.


In [ ]:
# sudo cp nodes.csv /var/lib/neo4j/import/
# sudo cp edges.csv /var/lib/neo4j/import/
# cypher-shell -u neo4j -p <your_password>
# LOAD CSV WITH HEADERS FROM 'file:///nodes.csv' AS row
# CREATE (:Entity {id: row.id, label: row.label});
# LOAD CSV WITH HEADERS FROM 'file:///edges.csv' AS row
# MATCH (a:Entity {id: row.source}), (b:Entity {id: row.target})
# CREATE (a)-[:RELATIONSHIP {type: row.relationship}]->(b);
# ALL DATABASE
# MATCH (n)-[r]->(m) RETURN n, r, m;
# QUERY RELATIONSHIPS
# MATCH (n)-[r]->(m) RETURN r LIMIT 10;
# QUERY PATIENT ALL DIAGNOSES
# MATCH (patient:Entity {id: "Patient ID: 10000980"})-[:RELATIONSHIP{type: "has"}]->(diagnosis:Entity)
# RETURN diagnosis.id AS diagnosis_id;
# QUERY ALL THE MEDICATION
# MATCH (patient:Entity {id: "Patient ID: 10000980"})-[:RELATIONSHIP{type: "receives"}]->(medication:Entity)
# RETURN medication.id AS medication_id
# QUERY ALL THE MEDICATION IN THE DB
# MATCH (patient:Entity)-[:RELATIONSHIP {type: "receives"}]->(medication:Entity)
# RETURN medication.id AS medication_id;

[sudo] password for andres: 
[sudo] password for andres: 